# Spark Quickstart Notebook

A self-contained tour of PySpark running in local mode: DataFrames, aggregations, Spark SQL, window functions, and a classic RDD word count.

**Requirements:** Java 17+ and `pip install -r requirements.txt`.

## 1. Start a SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("spark-quickstart")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.2.0


## 2. Create a DataFrame

In [2]:
sales = spark.createDataFrame(
    [
        ("2026-01-03", "North", "Laptop",  2, 1200.0),
        ("2026-01-05", "South", "Phone",   5,  650.0),
        ("2026-01-07", "North", "Tablet",  3,  400.0),
        ("2026-01-11", "East",  "Laptop",  1, 1250.0),
        ("2026-01-14", "South", "Laptop",  4, 1180.0),
        ("2026-01-18", "East",  "Phone",   7,  620.0),
        ("2026-01-21", "North", "Phone",   2,  640.0),
        ("2026-01-25", "West",  "Tablet",  6,  380.0),
        ("2026-01-28", "West",  "Laptop",  2, 1220.0),
        ("2026-01-30", "East",  "Tablet",  4,  390.0),
    ],
    schema="order_date STRING, region STRING, product STRING, qty INT, unit_price DOUBLE",
).withColumn("order_date", F.to_date("order_date"))

sales.printSchema()
sales.show()

root
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)



+----------+------+-------+---+----------+
|order_date|region|product|qty|unit_price|
+----------+------+-------+---+----------+
|2026-01-03| North| Laptop|  2|    1200.0|
|2026-01-05| South|  Phone|  5|     650.0|
|2026-01-07| North| Tablet|  3|     400.0|
|2026-01-11|  East| Laptop|  1|    1250.0|
|2026-01-14| South| Laptop|  4|    1180.0|
|2026-01-18|  East|  Phone|  7|     620.0|
|2026-01-21| North|  Phone|  2|     640.0|
|2026-01-25|  West| Tablet|  6|     380.0|
|2026-01-28|  West| Laptop|  2|    1220.0|
|2026-01-30|  East| Tablet|  4|     390.0|
+----------+------+-------+---+----------+



## 3. Transformations

In [3]:
enriched = (
    sales
    .withColumn("revenue", F.col("qty") * F.col("unit_price"))
    .withColumn("size", F.when(F.col("revenue") >= 3000, "large").otherwise("small"))
)
enriched.filter(F.col("size") == "large").orderBy(F.desc("revenue")).show()

+----------+------+-------+---+----------+-------+-----+
|order_date|region|product|qty|unit_price|revenue| size|
+----------+------+-------+---+----------+-------+-----+
|2026-01-14| South| Laptop|  4|    1180.0| 4720.0|large|
|2026-01-18|  East|  Phone|  7|     620.0| 4340.0|large|
|2026-01-05| South|  Phone|  5|     650.0| 3250.0|large|
+----------+------+-------+---+----------+-------+-----+



## 4. Aggregations

In [4]:
by_region = (
    enriched.groupBy("region")
    .agg(
        F.count("*").alias("orders"),
        F.sum("qty").alias("units"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.avg("revenue"), 2).alias("avg_order"),
    )
    .orderBy(F.desc("total_revenue"))
)
by_region.show()

+------+------+-----+-------------+---------+
|region|orders|units|total_revenue|avg_order|
+------+------+-----+-------------+---------+
| South|     2|    9|       7970.0|   3985.0|
|  East|     3|   12|       7150.0|  2383.33|
| North|     3|    7|       4880.0|  1626.67|
|  West|     2|    8|       4720.0|   2360.0|
+------+------+-----+-------------+---------+



In [5]:
# Pivot: revenue per region x product
enriched.groupBy("region").pivot("product").sum("revenue").na.fill(0).orderBy("region").show()

+------+------+------+------+
|region|Laptop| Phone|Tablet|
+------+------+------+------+
|  East|1250.0|4340.0|1560.0|
| North|2400.0|1280.0|1200.0|
| South|4720.0|3250.0|   0.0|
|  West|2440.0|   0.0|2280.0|
+------+------+------+------+



## 5. Spark SQL

In [6]:
enriched.createOrReplaceTempView("sales")

spark.sql("""
    SELECT product,
           SUM(qty)                AS units,
           ROUND(SUM(revenue), 2)  AS revenue
    FROM sales
    GROUP BY product
    ORDER BY revenue DESC
""").show()

+-------+-----+-------+
|product|units|revenue|
+-------+-----+-------+
| Laptop|    9|10810.0|
|  Phone|   14| 8870.0|
| Tablet|   13| 5040.0|
+-------+-----+-------+



## 6. Window functions

Rank orders within each region and compute a running revenue total.

In [7]:
w_rank = Window.partitionBy("region").orderBy(F.desc("revenue"))
w_running = Window.partitionBy("region").orderBy("order_date").rowsBetween(Window.unboundedPreceding, 0)

(
    enriched
    .withColumn("rank_in_region", F.rank().over(w_rank))
    .withColumn("running_revenue", F.sum("revenue").over(w_running))
    .select("region", "order_date", "product", "revenue", "rank_in_region", "running_revenue")
    .orderBy("region", "order_date")
    .show()
)

+------+----------+-------+-------+--------------+---------------+
|region|order_date|product|revenue|rank_in_region|running_revenue|
+------+----------+-------+-------+--------------+---------------+
|  East|2026-01-11| Laptop| 1250.0|             3|         1250.0|
|  East|2026-01-18|  Phone| 4340.0|             1|         5590.0|
|  East|2026-01-30| Tablet| 1560.0|             2|         7150.0|
| North|2026-01-03| Laptop| 2400.0|             1|         2400.0|
| North|2026-01-07| Tablet| 1200.0|             3|         3600.0|
| North|2026-01-21|  Phone| 1280.0|             2|         4880.0|
| South|2026-01-05|  Phone| 3250.0|             2|         3250.0|
| South|2026-01-14| Laptop| 4720.0|             1|         7970.0|
|  West|2026-01-25| Tablet| 2280.0|             2|         2280.0|
|  West|2026-01-28| Laptop| 2440.0|             1|         4720.0|
+------+----------+-------+-------+--------------+---------------+



## 7. RDD API: word count

In [8]:
text = [
    "spark makes big data simple",
    "big data needs fast processing",
    "spark is fast and spark is fun",
]

counts = (
    spark.sparkContext.parallelize(text)
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda kv: (-kv[1], kv[0]))
)
for word, n in counts.take(5):
    print(f"{word:<12}{n}")

spark       3
big         2
data        2
fast        2
is          2


## 8. Inspect the query plan

In [9]:
by_region.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#48 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#48 DESC NULLS LAST, 4), ENSURE_REQUIREMENTS, [plan_id=490]
      +- HashAggregate(keys=[region#1], functions=[count(1), sum(qty#3), sum(revenue#22), avg(revenue#22)])
         +- Exchange hashpartitioning(region#1, 4), ENSURE_REQUIREMENTS, [plan_id=487]
            +- HashAggregate(keys=[region#1], functions=[partial_count(1), partial_sum(qty#3), partial_sum(revenue#22), partial_avg(revenue#22)])
               +- Project [region#1, qty#3, (cast(qty#3 as double) * unit_price#4) AS revenue#22]
                  +- Scan ExistingRDD[order_date#0,region#1,product#2,qty#3,unit_price#4]




## 9. Clean up

In [10]:
spark.stop()